# Music Annotation Project Notebook 63
## Frozen 59R2 diagnostic operating-point envelope across 151 reviewed pages

**Runtime:** CPU. In Colab, select **Runtime → Run all**.

This notebook performs a read-only, post-outcome sensitivity audit using the durable Notebook 61 pixel probabilities. It does **not** retrain, refit, recalibrate, edit the frozen 59R2 bundle, or claim that any alternative threshold is 59R2.

### Diagnostic operating points
`0.995, 0.993, 0.990, 0.985, 0.980, 0.975, 0.950`

At each operating point, the notebook applies the same frozen page-global coherence mechanism, changing only the diagnostic pixel/region threshold together. It reports:

- removal coverage by pixel, page and component;
- incremental coverage relative to frozen 0.995;
- concentration in pages, source groups and components;
- page/source stability and rank changes;
- observed-intensity and paper-delta strata;
- comparison with the persisted 59R1 decision;
- available truth-side safety, but only if an explicitly compatible truth table can be resolved and joined without ambiguity;
- Page258 renderings and cohort review queues;
- a curated upload bundle excluding the large source parquet.

### Interpretation boundary
These are **diagnostic policies**, not frozen challengers. Lower thresholds are not authorised for independent or production use. Clean truth, where available, is used only after decisions are assembled and is never used to create diagnostic decisions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib

## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math, platform, shutil, sys, zipfile

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
NB61_ROOT = ROOT / '59r2_annotated_only_shadow_test'
NB61_RUN_OVERRIDE = None
R59R2_OVERRIDE = None

THRESHOLDS = [0.995, 0.993, 0.990, 0.985, 0.980, 0.975, 0.950]
EXPECTED_PAGES = 151
FOCUS_PAGE = 'Page258'
TOP_N_CONCENTRATION = 10
TOP_REVIEW_PAGES = 20
OVERLAY_ALPHA = 0.55

# Optional metadata/truth overrides. Leave None for fail-closed automatic discovery.
SOURCE_MANIFEST_OVERRIDE = None
TRUTH_TABLE_OVERRIDE = None
TRUTH_SEMANTICS_OVERRIDE = None  # e.g. {'label_column':'truth','print_values':[1],'annotation_values':[0],'uncertain_values':[2]}

OUTPUT_ROOT = ROOT / 'notebook63_59r2_diagnostic_operating_point_envelope'
RUN = OUTPUT_ROOT / (datetime.now().strftime('%Y%m%d_%H%M%S') + '_notebook63_59r2_threshold_envelope')
TABLES, CHARTS, FOCUS, REVIEW, AUDIT, UPLOAD = [RUN/x for x in ['tables','charts','page258','review','audit','upload']]
for p in [RUN,TABLES,CHARTS,FOCUS,REVIEW,AUDIT,UPLOAD]: p.mkdir(parents=True,exist_ok=False)

print('Output:', RUN)

## 2. Helpers and exact source discovery

In [ ]:
def require(v,msg):
    if not v: raise RuntimeError(msg)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def atomic_json(path,value):
    tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str)); tmp.replace(path)

def latest_complete(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def wilson_upper(errors,n,z=1.96):
    if n==0:return np.nan
    p=errors/n; d=1+z*z/n; c=p+z*z/(2*n); r=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n)); return (c+r)/d

def gini(values):
    x=np.sort(np.asarray(values,dtype=np.float64)); x=x[x>=0]
    if len(x)==0 or x.sum()==0:return 0.0
    return float((2*np.sum((np.arange(1,len(x)+1))*x)/(len(x)*x.sum()))-(len(x)+1)/len(x))

def hhi(values):
    x=np.asarray(values,dtype=np.float64); s=x.sum()
    return float(np.sum((x/s)**2)) if s>0 else 0.0

def top_share(values,n=TOP_N_CONCENTRATION):
    x=np.sort(np.asarray(values,dtype=np.float64))[::-1]; return float(x[:n].sum()/x.sum()) if x.sum()>0 else 0.0

NB61_REQUIRED=['tables/59r2_scored_reviewed_cohort.parquet','experiment_identity.json',f'pages/{FOCUS_PAGE}/observed.png',f'pages/{FOCUS_PAGE}/59r1_shadow.png',f'pages/{FOCUS_PAGE}/59r2_shadow.png']
NB61=Path(NB61_RUN_OVERRIDE) if NB61_RUN_OVERRIDE else latest_complete(NB61_ROOT,NB61_REQUIRED)
require(NB61 is not None,'Complete Notebook 61 run not found. Set NB61_RUN_OVERRIDE.')
PARQUET_PATH=NB61/'tables'/'59r2_scored_reviewed_cohort.parquet'
IDENTITY61=json.loads((NB61/'experiment_identity.json').read_text())

if R59R2_OVERRIDE is not None: R59R2=Path(R59R2_OVERRIDE)
elif IDENTITY61.get('r59r2') and Path(IDENTITY61['r59r2']).exists(): R59R2=Path(IDENTITY61['r59r2'])
else: R59R2=latest_complete(ROOT,['bundle/policy.json','bundle/runtime_contract.json'])
require(R59R2 is not None,'Frozen 59R2 bundle not found. Set R59R2_OVERRIDE.')

print('Notebook 61:',NB61)
print('Frozen 59R2:',R59R2)

## 3. Verify the frozen coherence contract

In [ ]:
BUNDLE=R59R2/'bundle'; policy=json.loads((BUNDLE/'policy.json').read_text()); contract=json.loads((BUNDLE/'runtime_contract.json').read_text())
def first(*xs):
    for x in xs:
        if x is not None:return x
    return None
FROZEN_THRESHOLD=float(first(contract.get('threshold'),contract.get('remove_threshold'),policy.get('threshold'),policy.get('remove_threshold')))
CONNECTIVITY=int(first(contract.get('connectivity'),policy.get('connectivity'),8))
MINIMUM_RUN=int(first(contract.get('minimum_run'),contract.get('min_run'),policy.get('minimum_run'),2))
FROZEN_REGION_MEAN=float(first(contract.get('region_mean_threshold'),policy.get('region_mean_threshold'),FROZEN_THRESHOLD))
require(abs(FROZEN_THRESHOLD-0.995)<1e-12,f'Expected frozen threshold 0.995; found {FROZEN_THRESHOLD}')
require(CONNECTIVITY in [4,8],'Connectivity must be 4 or 8.')
contract_record={'policy_id':policy.get('policy_id'),'compatibility_id':policy.get('compatibility_id'),'frozen_threshold':FROZEN_THRESHOLD,'connectivity':CONNECTIVITY,'minimum_run':MINIMUM_RUN,'frozen_region_mean_threshold':FROZEN_REGION_MEAN,'diagnostic_rule':'pixel threshold and region mean threshold move together; connectivity/minimum run remain frozen','thresholds':THRESHOLDS,'policy_sha256':sha256_file(BUNDLE/'policy.json'),'runtime_contract_sha256':sha256_file(BUNDLE/'runtime_contract.json')}
atomic_json(AUDIT/'resolved_frozen_contract.json',contract_record)
print(json.dumps(contract_record,indent=2))

## 4. Load the compact 151-page scored cohort

In [ ]:
cols=['page','y','x','component_key','safe_remove','p_remove','p_remove_59r2','paper_value']
try: cohort=pd.read_parquet(PARQUET_PATH,columns=cols)
except Exception:
    cohort=pd.read_parquet(PARQUET_PATH)
required=set(cols); missing=required-set(cohort.columns); require(not missing,f'Parquet lacks: {sorted(missing)}')
cohort=cohort[cols].copy(); cohort['page']=cohort.page.astype(str)
for c in ['y','x','component_key']: cohort[c]=pd.to_numeric(cohort[c],errors='raise').astype(np.int32)
for c in ['p_remove','p_remove_59r2','paper_value']: cohort[c]=pd.to_numeric(cohort[c],errors='raise').astype(np.float32)
cohort['safe_remove']=cohort.safe_remove.astype(bool)
require(not cohort.duplicated(['page','y','x']).any(),'Duplicate page/y/x rows.')
require(np.isfinite(cohort[['p_remove','p_remove_59r2','paper_value']].to_numpy()).all(),'Non-finite compact cohort values.')
require(cohort.page.nunique()==EXPECTED_PAGES,f'Expected {EXPECTED_PAGES} pages, found {cohort.page.nunique()}.')
print('Rows:',f'{len(cohort):,}','Pages:',cohort.page.nunique(),'Components:',cohort[['page','component_key']].drop_duplicates().shape[0])

## 5. Resolve page images, observed intensity, source groups and available truth

In [ ]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def image_map(folder): return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}

def resolve_observed(page):
    candidates=[NB61/'pages'/page/'observed.png']
    r60=IDENTITY61.get('r60')
    if r60:
        r60=Path(r60); candidates += [r60/'pages'/page/'observed.png',r60/'whole_pages'/page/'observed.png']
    candidates += [ROOT/'aligned'/f'{page}.png',ROOT/'source'/'annotated_only'/f'{page}.png']
    for p in candidates:
        if p.is_file():return p
    for folder in [ROOT/'aligned',ROOT/'source'/'annotated_only']:
        q=image_map(folder).get(page)
        if q is not None:return q
    return None

page_paths={p:resolve_observed(p) for p in sorted(cohort.page.unique())}
require(not [p for p,v in page_paths.items() if v is None],'One or more observed pages could not be resolved.')
obs_values=np.empty(len(cohort),np.uint8); dims=[]
for page,idx in cohort.groupby('page',sort=False).groups.items():
    im=cv2.imread(str(page_paths[page]),cv2.IMREAD_GRAYSCALE); require(im is not None,f'Unreadable observed image: {page_paths[page]}')
    ii=np.asarray(list(idx),dtype=np.int64); yy=cohort.loc[ii,'y'].to_numpy(); xx=cohort.loc[ii,'x'].to_numpy(); require((yy>=0).all() and (xx>=0).all() and (yy<im.shape[0]).all() and (xx<im.shape[1]).all(),f'{page}: coordinate outside image')
    obs_values[ii]=im[yy,xx]; dims.append({'page':page,'height':im.shape[0],'width':im.shape[1],'observed_path':str(page_paths[page]),'observed_sha256':sha256_file(page_paths[page])})
cohort['observed_value']=obs_values
cohort['paper_minus_observed']=cohort.paper_value.astype(np.float32)-cohort.observed_value.astype(np.float32)
pd.DataFrame(dims).to_csv(AUDIT/'resolved_page_images.csv',index=False)

# Source groups: prefer an explicit override, then run-local/input manifests, then project manifest.
manifest_candidates=[]
if SOURCE_MANIFEST_OVERRIDE: manifest_candidates=[Path(SOURCE_MANIFEST_OVERRIDE)]
else:
    r60=Path(IDENTITY61['r60']) if IDENTITY61.get('r60') else None
    manifest_candidates=[NB61/'inputs'/'development_score_manifest.csv',ROOT/'development_score_manifest.csv']
    if r60: manifest_candidates=[r60/'inputs'/'development_score_manifest.csv',r60/'tables'/'page_manifest.csv']+manifest_candidates
source_manifest=None
for p in manifest_candidates:
    if p and p.is_file():
        q=pd.read_csv(p,dtype=str)
        if {'page','source_group'}.issubset(q.columns) and not q.page.duplicated().any(): source_manifest=(p,q[['page','source_group']].copy()); break
if source_manifest:
    source_path,sm=source_manifest; sm['page']=sm.page.astype(str); cohort=cohort.merge(sm,on='page',how='left',validate='many_to_one'); source_status={'available':True,'path':str(source_path),'sha256':sha256_file(source_path),'mapped_pages':int(cohort.loc[cohort.source_group.notna(),'page'].nunique()),'missing_pages':sorted(cohort.loc[cohort.source_group.isna(),'page'].unique().tolist())}
else:
    cohort['source_group']='UNAVAILABLE'; source_status={'available':False,'reason':'No unambiguous page/source_group manifest resolved. Source-group metrics will be labelled unavailable.'}
if 'source_group' in cohort: cohort['source_group']=cohort.source_group.fillna('UNAVAILABLE').astype(str)
atomic_json(AUDIT/'source_group_resolution.json',source_status)

# Truth is optional and fail-closed. Only exact page/y/x tables with declared semantics are accepted.
truth_status={'available':False,'reason':'No compatible truth table resolved.'}; truth=None
truth_candidates=[]
if TRUTH_TABLE_OVERRIDE: truth_candidates=[Path(TRUTH_TABLE_OVERRIDE)]
else:
    r60=Path(IDENTITY61['r60']) if IDENTITY61.get('r60') else None
    candidates=[ROOT/'truth'/'pixel_truth.parquet',ROOT/'tables'/'pixel_truth.parquet']
    if r60: candidates=[r60/'tables'/'pixel_truth.parquet',r60/'tables'/'all_eligible_truth.parquet']+candidates
    truth_candidates=[p for p in candidates if p.is_file()]
for p in truth_candidates:
    try: q=pd.read_parquet(p) if p.suffix.lower()=='.parquet' else pd.read_csv(p)
    except Exception: continue
    if not {'page','y','x'}.issubset(q.columns): continue
    semantics=TRUTH_SEMANTICS_OVERRIDE
    if semantics is None:
        if 'truth' in q.columns and set(pd.to_numeric(q.truth,errors='coerce').dropna().unique()).issubset({0,1,2}): semantics={'label_column':'truth','print_values':[1],'annotation_values':[0],'uncertain_values':[2]}
        elif 'truth_label' in q.columns: semantics={'label_column':'truth_label','print_values':['PRINT'],'annotation_values':['ANNOTATION'],'uncertain_values':['UNCERTAIN']}
    if semantics is None: continue
    label=semantics['label_column']; needed=['page','y','x',label]; q=q[needed].copy(); q['page']=q.page.astype(str); q['y']=pd.to_numeric(q.y,errors='raise').astype(np.int32); q['x']=pd.to_numeric(q.x,errors='raise').astype(np.int32)
    if q.duplicated(['page','y','x']).any(): continue
    q['truth_class']=np.select([q[label].isin(semantics['print_values']),q[label].isin(semantics['annotation_values']),q[label].isin(semantics.get('uncertain_values',[]))],['PRINT','ANNOTATION','UNCERTAIN'],default='UNMAPPED')
    if (q.truth_class=='UNMAPPED').any(): continue
    truth=(p,q[['page','y','x','truth_class']]); truth_status={'available':True,'path':str(p),'sha256':sha256_file(p),'semantics':semantics,'rows':len(q)}; break
atomic_json(AUDIT/'truth_resolution.json',truth_status)
print('Source groups:',source_status)
print('Truth:',truth_status)

## 6. Apply each diagnostic operating point page-globally

In [ ]:
def coherent_decision(y,x,probability,height,width,threshold,connectivity,minimum_run):
    y=np.asarray(y,np.int32); x=np.asarray(x,np.int32); probability=np.asarray(probability,np.float32)
    provisional=probability>=float(threshold)
    candidate=np.zeros((height,width),np.uint8); candidate[y[provisional],x[provisional]]=1
    n,labels,stats,_=cv2.connectedComponentsWithStats(candidate,int(connectivity))
    pmap=np.full((height,width),np.nan,np.float32); pmap[y,x]=probability
    safe=np.zeros((height,width),bool)
    for cid in range(1,n):
        region=labels==cid; size=int(stats[cid,cv2.CC_STAT_AREA]); vals=pmap[region]; finite=vals[np.isfinite(vals)]
        if size>=int(minimum_run) and len(finite)==size and float(finite.mean())>=float(threshold) and float(finite.min())>=float(threshold): safe[region]=True
    return safe[y,x],provisional

dim_map=pd.DataFrame(dims).set_index('page')[['height','width']].to_dict('index')
page_array=cohort.page.to_numpy(); y_all=cohort.y.to_numpy(); x_all=cohort.x.to_numpy(); p_all=cohort.p_remove_59r2.to_numpy()
for t in THRESHOLDS:
    final=np.zeros(len(cohort),bool); provisional=np.zeros(len(cohort),bool)
    for page,idx in cohort.groupby('page',sort=False).groups.items():
        ii=np.asarray(list(idx),dtype=np.int64); d=dim_map[page]; f,p=coherent_decision(y_all[ii],x_all[ii],p_all[ii],d['height'],d['width'],t,CONNECTIVITY,MINIMUM_RUN); final[ii]=f; provisional[ii]=p
    cohort[f'provisional_{t:.3f}']=provisional; cohort[f'safe_{t:.3f}']=final

# Exact frozen replay against Notebook 61 aggregate and Page258 shadow.
require(np.array_equal(cohort[f'safe_{FROZEN_THRESHOLD:.3f}'].to_numpy()[cohort.page==FOCUS_PAGE],
                       cohort.loc[cohort.page==FOCUS_PAGE,f'safe_{FROZEN_THRESHOLD:.3f}'].to_numpy()),'Internal replay alignment failure.')
if (NB61/'shadow_test_summary.json').is_file():
    s61=json.loads((NB61/'shadow_test_summary.json').read_text()); require(int(cohort[f'safe_{FROZEN_THRESHOLD:.3f}'].sum())==int(s61['59r2_safe_remove']),'Frozen 0.995 aggregate does not reproduce Notebook 61.')
atomic_json(AUDIT/'frozen_replay.json',{'passed':True,'frozen_selected':int(cohort[f'safe_{FROZEN_THRESHOLD:.3f}'].sum()),'expected_from_notebook61':int(s61['59r2_safe_remove']) if 's61' in globals() else None})
print('Frozen replay selected:',f'{cohort[f"safe_{FROZEN_THRESHOLD:.3f}"].sum():,}')

## 7. Coverage, concentration and page/source stability

In [ ]:
summary_rows=[]; page_tables=[]; group_tables=[]; component_tables=[]
frozen_col=f'safe_{FROZEN_THRESHOLD:.3f}'
for t in THRESHOLDS:
    col=f'safe_{t:.3f}'; selected=cohort[col]
    by_page=cohort.assign(selected=selected).groupby(['page','source_group'],as_index=False).agg(candidate_pixels=('x','size'),selected_pixels=('selected','sum'),components_total=('component_key','nunique'))
    by_page['selected_fraction']=by_page.selected_pixels/by_page.candidate_pixels; by_page['threshold']=t; by_page['delta_vs_frozen']=by_page.selected_pixels-by_page.merge(by_page[['page']].assign(dummy=0),on='page').dummy # overwritten below
    frozen_page=cohort.assign(selected=cohort[frozen_col]).groupby('page').selected.sum()
    by_page['delta_vs_frozen']=by_page.selected_pixels-by_page.page.map(frozen_page).astype(int)
    page_tables.append(by_page)

    selected_components=(cohort[selected].groupby(['page','source_group','component_key'],as_index=False).size().rename(columns={'size':'selected_pixels'}))
    all_components=cohort.groupby(['page','source_group','component_key'],as_index=False).size().rename(columns={'size':'candidate_pixels'})
    ct=all_components.merge(selected_components,on=['page','source_group','component_key'],how='left'); ct['selected_pixels']=ct.selected_pixels.fillna(0).astype(int); ct['selected_fraction']=ct.selected_pixels/ct.candidate_pixels; ct['threshold']=t; component_tables.append(ct)

    gt=by_page.groupby('source_group',as_index=False).agg(pages=('page','nunique'),candidate_pixels=('candidate_pixels','sum'),selected_pixels=('selected_pixels','sum'),pages_selected=('selected_pixels',lambda s:int((s>0).sum())))
    gt['selected_fraction']=gt.selected_pixels/gt.candidate_pixels; gt['threshold']=t; group_tables.append(gt)

    page_vals=by_page.selected_pixels.to_numpy(); comp_vals=ct.selected_pixels.to_numpy(); group_vals=gt.selected_pixels.to_numpy()
    inc=int((selected&~cohort[frozen_col]).sum())
    summary_rows.append({'threshold':t,'candidate_pixels':len(cohort),'selected_pixels':int(selected.sum()),'coverage':float(selected.mean()),'incremental_pixels_vs_0_995':inc,'pages_selected':int((by_page.selected_pixels>0).sum()),'pages_with_100_or_more':int((by_page.selected_pixels>=100).sum()),'median_selected_per_page':float(by_page.selected_pixels.median()),'p90_selected_per_page':float(by_page.selected_pixels.quantile(.90)),'max_selected_page':str(by_page.sort_values('selected_pixels').iloc[-1].page),'max_selected_page_pixels':int(by_page.selected_pixels.max()),'page_gini':gini(page_vals),'top10_page_share':top_share(page_vals),'page_hhi':hhi(page_vals),'selected_components':int((ct.selected_pixels>0).sum()),'components_100_or_more':int((ct.selected_pixels>=100).sum()),'component_gini':gini(comp_vals),'top10_component_share':top_share(comp_vals),'component_hhi':hhi(comp_vals),'source_groups_available':bool(source_status.get('available')),'source_groups_selected':int((gt.selected_pixels>0).sum()) if source_status.get('available') else None,'source_group_gini':gini(group_vals) if source_status.get('available') else None,'top3_source_group_share':top_share(group_vals,3) if source_status.get('available') else None,'observed_ge_180_selected':int((selected&(cohort.observed_value>=180)).sum()),'observed_ge_210_selected':int((selected&(cohort.observed_value>=210)).sum()),'mean_paper_minus_observed_selected':float(cohort.loc[selected,'paper_minus_observed'].mean()) if selected.any() else np.nan,'agreement_with_59r1':float((selected==cohort.safe_remove).mean()),'59r1_selected_recovered':int((selected&cohort.safe_remove).sum()),'59r1_only_remaining':int((~selected&cohort.safe_remove).sum()),'59r2_only_vs_59r1':int((selected&~cohort.safe_remove).sum())})

summary=pd.DataFrame(summary_rows); page_df=pd.concat(page_tables,ignore_index=True); group_df=pd.concat(group_tables,ignore_index=True); component_df=pd.concat(component_tables,ignore_index=True)
summary.to_csv(TABLES/'threshold_envelope_summary.csv',index=False); page_df.to_csv(TABLES/'threshold_results_by_page.csv',index=False); group_df.to_csv(TABLES/'threshold_results_by_source_group.csv',index=False); component_df.to_csv(TABLES/'threshold_results_by_component.csv',index=False)

display(summary)

## 8. Stability transitions and threshold concentration diagnostics

In [ ]:
transitions=[]
for higher,lower in zip(THRESHOLDS[:-1],THRESHOLDS[1:]):
    hc=f'safe_{higher:.3f}'; lc=f'safe_{lower:.3f}'; newly=cohort[lc]&~cohort[hc]
    page_new=cohort.assign(newly=newly).groupby('page').newly.sum()
    comp_new=cohort.assign(newly=newly).groupby(['page','component_key']).newly.sum()
    transitions.append({'from_threshold':higher,'to_threshold':lower,'new_pixels':int(newly.sum()),'pages_gaining':int((page_new>0).sum()),'components_gaining':int((comp_new>0).sum()),'top10_page_share_of_increment':top_share(page_new.to_numpy()),'top10_component_share_of_increment':top_share(comp_new.to_numpy()),'observed_ge_180':int((newly&(cohort.observed_value>=180)).sum()),'observed_ge_210':int((newly&(cohort.observed_value>=210)).sum()),'59r1_selected_increment':int((newly&cohort.safe_remove).sum()),'not_59r1_increment':int((newly&~cohort.safe_remove).sum())})
transitions=pd.DataFrame(transitions); transitions.to_csv(TABLES/'incremental_threshold_transitions.csv',index=False)

# Spearman rank stability of page coverage relative to frozen.
pivot=page_df.pivot(index='page',columns='threshold',values='selected_pixels')
stability=[]
for t in THRESHOLDS:
    stability.append({'threshold':t,'spearman_vs_0_995':float(pivot[t].rank().corr(pivot[0.995].rank())),'pearson_vs_0_995':float(pivot[t].corr(pivot[0.995])),'pages_newly_nonzero':int(((pivot[t]>0)&(pivot[0.995]==0)).sum())})
stability=pd.DataFrame(stability); stability.to_csv(TABLES/'page_rank_stability.csv',index=False)
display(transitions); display(stability)

## 9. Available truth-side safety, fail closed if truth is unavailable

In [ ]:
truth_summary=[]; truth_page=pd.DataFrame(); truth_group=pd.DataFrame(); truth_component=pd.DataFrame()
if truth is not None:
    truth_path,tq=truth
    scored=cohort.merge(tq,on=['page','y','x'],how='left',validate='one_to_one')
    matched=int(scored.truth_class.notna().sum()); atomic_json(AUDIT/'truth_join_audit.json',{'truth_path':str(truth_path),'cohort_rows':len(scored),'matched_rows':matched,'matched_fraction':matched/len(scored),'unmatched_rows':len(scored)-matched})
    for t in THRESHOLDS:
        col=f'safe_{t:.3f}'; sel=scored[col]&scored.truth_class.notna(); tp=int((sel&(scored.truth_class=='ANNOTATION')).sum()); fp=int((sel&(scored.truth_class=='PRINT')).sum()); u=int((sel&(scored.truth_class=='UNCERTAIN')).sum()); n=tp+fp
        truth_summary.append({'threshold':t,'truth_matched_pixels':matched,'selected_truth_scored':int(sel.sum()),'correct_removal_annotation':tp,'false_remove_print':fp,'truth_uncertain_selected':u,'false_remove_rate':fp/max(n,1),'wilson95_upper_false_remove':wilson_upper(fp,n),'annotation_recall_within_truth':tp/max(int((scored.truth_class=='ANNOTATION').sum()),1)})
    truth_summary=pd.DataFrame(truth_summary); truth_summary.to_csv(TABLES/'truth_safety_summary.csv',index=False)
    rows=[]
    for t in THRESHOLDS:
        col=f'safe_{t:.3f}'; q=scored[scored.truth_class.notna()].copy(); q['selected']=q[col]
        z=q.groupby(['page','source_group'],as_index=False).apply(lambda g:pd.Series({'selected':int(g.selected.sum()),'correct_removal_annotation':int((g.selected&(g.truth_class=='ANNOTATION')).sum()),'false_remove_print':int((g.selected&(g.truth_class=='PRINT')).sum()),'truth_uncertain_selected':int((g.selected&(g.truth_class=='UNCERTAIN')).sum())}),include_groups=False).reset_index(drop=True); z['threshold']=t; rows.append(z)
    truth_page=pd.concat(rows,ignore_index=True); truth_page.to_csv(TABLES/'truth_safety_by_page.csv',index=False)
    truth_group=truth_page.groupby(['threshold','source_group'],as_index=False).agg(pages=('page','nunique'),selected=('selected','sum'),correct_removal_annotation=('correct_removal_annotation','sum'),false_remove_print=('false_remove_print','sum'),truth_uncertain_selected=('truth_uncertain_selected','sum')); truth_group['false_remove_rate']=truth_group.false_remove_print/np.maximum(truth_group.correct_removal_annotation+truth_group.false_remove_print,1); truth_group.to_csv(TABLES/'truth_safety_by_source_group.csv',index=False)
    display(truth_summary)
else:
    pd.DataFrame([{'truth_available':False,'reason':truth_status.get('reason'),'interpretation':'Coverage and stability can be measured; safety cannot be established from the compact reviewed-cohort parquet alone.'}]).to_csv(TABLES/'truth_safety_unavailable.csv',index=False)
    print('TRUTH-SIDE SAFETY UNAVAILABLE:',truth_status.get('reason'))

## 10. Charts: coverage, concentration, stability and 59R1 recovery

In [ ]:
fig,ax=plt.subplots(2,2,figsize=(14,10))
q=summary.sort_values('threshold')
ax[0,0].plot(q.threshold,q.selected_pixels,marker='o'); ax[0,0].set(title='Removal coverage',xlabel='Diagnostic threshold',ylabel='SAFE_REMOVE pixels')
ax[0,1].plot(q.threshold,q.top10_page_share,marker='o',label='top 10 pages'); ax[0,1].plot(q.threshold,q.top10_component_share,marker='o',label='top 10 components'); ax[0,1].set(title='Concentration of selected pixels',xlabel='Diagnostic threshold',ylabel='Share'); ax[0,1].legend()
ax[1,0].plot(stability.threshold,stability.spearman_vs_0_995,marker='o'); ax[1,0].set(title='Page-rank stability versus frozen 0.995',xlabel='Diagnostic threshold',ylabel='Spearman correlation',ylim=(-.05,1.05))
ax[1,1].plot(q.threshold,q['59r1_selected_recovered'],marker='o',label='59R1 selected recovered'); ax[1,1].plot(q.threshold,q['59r1_only_remaining'],marker='o',label='59R1-only remaining'); ax[1,1].set(title='Relationship to persisted 59R1',xlabel='Diagnostic threshold',ylabel='Pixels'); ax[1,1].legend()
for a in ax.flat:a.invert_xaxis();a.grid(alpha=.25)
plt.tight_layout(); plt.savefig(CHARTS/'threshold_envelope_dashboard.png',dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)

fig,ax=plt.subplots(figsize=(12,7));
for t in THRESHOLDS:
    q=page_df[page_df.threshold==t].sort_values('selected_pixels',ascending=False).reset_index(drop=True); ax.plot(np.arange(1,len(q)+1),q.selected_pixels.cumsum()/max(q.selected_pixels.sum(),1),label=f'{t:.3f}')
ax.set(title='Cumulative page concentration',xlabel='Pages ordered by selected pixels',ylabel='Cumulative share'); ax.legend(); ax.grid(alpha=.25); plt.tight_layout(); plt.savefig(CHARTS/'page_concentration_curves.png',dpi=180); plt.close(fig)

if truth is not None:
    fig,ax=plt.subplots(1,2,figsize=(13,5)); tq=truth_summary.sort_values('threshold'); ax[0].plot(tq.threshold,tq.correct_removal_annotation,marker='o',label='correct annotation removal'); ax[0].plot(tq.threshold,tq.false_remove_print,marker='o',label='false printed removal'); ax[0].invert_xaxis(); ax[0].legend(); ax[0].set(title='Truth-side selected pixels',xlabel='Threshold',ylabel='Pixels'); ax[1].plot(tq.threshold,tq.false_remove_rate,marker='o',label='observed'); ax[1].plot(tq.threshold,tq.wilson95_upper_false_remove,marker='o',label='Wilson 95% upper'); ax[1].invert_xaxis(); ax[1].legend(); ax[1].set(title='Truth-side false-remove risk',xlabel='Threshold',ylabel='Rate'); plt.tight_layout(); plt.savefig(CHARTS/'truth_safety_dashboard.png',dpi=180); plt.close(fig)

## 11. Page258 threshold ladder and cohort review queues

In [ ]:
def blend(gray,mask,bgr=(0,0,255),alpha=OVERLAY_ALPHA):
    base=cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR); colour=base.copy(); colour[mask]=bgr; out=base.copy(); out[mask]=cv2.addWeighted(base,1-alpha,colour,alpha,0)[mask]; return out

focus_rows=cohort[cohort.page==FOCUS_PAGE].copy(); observed=cv2.imread(str(page_paths[FOCUS_PAGE]),0); fy=focus_rows.y.to_numpy(); fx=focus_rows.x.to_numpy(); fpaper=focus_rows.paper_value.to_numpy()
fig,axes=plt.subplots(2,4,figsize=(18,10)); axes=axes.flat
for i,t in enumerate(THRESHOLDS):
    safe=focus_rows[f'safe_{t:.3f}'].to_numpy(); out=observed.copy(); out[fy[safe],fx[safe]]=np.clip(np.rint(fpaper[safe]),0,255).astype(np.uint8); cv2.imwrite(str(FOCUS/f'page258_threshold_{t:.3f}.png'),out)
    mask=np.zeros(observed.shape,bool); mask[fy[safe],fx[safe]]=True; overlay=blend(observed,mask,(0,0,255)); cv2.imwrite(str(FOCUS/f'page258_threshold_{t:.3f}_overlay.png'),overlay)
    axes[i].imshow(out,cmap='gray',vmin=0,vmax=255); axes[i].set_title(f'{t:.3f}: {safe.sum():,} removed'); axes[i].axis('off')
axes[7].axis('off'); plt.tight_layout(); plt.savefig(FOCUS/'page258_threshold_ladder.png',dpi=150,bbox_inches='tight'); plt.close(fig)

# Review queues per threshold: most selected, largest increment, and highest concentration components.
queues=[]
frozen_page=page_df[page_df.threshold==0.995].set_index('page').selected_pixels
for t in THRESHOLDS:
    q=page_df[page_df.threshold==t].copy(); q['increment_vs_frozen']=q.selected_pixels-q.page.map(frozen_page).fillna(0)
    for reason,sortcol in [('most_selected','selected_pixels'),('largest_increment_vs_frozen','increment_vs_frozen')]:
        z=q.sort_values(sortcol,ascending=False).head(TOP_REVIEW_PAGES).copy(); z['review_reason']=reason; queues.append(z)
review_queue=pd.concat(queues,ignore_index=True).drop_duplicates(['threshold','page','review_reason']); review_queue.to_csv(TABLES/'visual_review_queue.csv',index=False)

# Compact threshold ladder for the top pages at 0.985 incremental gain.
top_pages=page_df[page_df.threshold==0.985].assign(increment=lambda d:d.selected_pixels-d.page.map(frozen_page).fillna(0)).sort_values('increment',ascending=False).head(12).page.tolist()
fig,axes=plt.subplots(len(top_pages),len(THRESHOLDS),figsize=(4*len(THRESHOLDS),5.2*len(top_pages)),squeeze=False)
for r,page in enumerate(top_pages):
    im=cv2.imread(str(page_paths[page]),0); g=cohort[cohort.page==page]; yy=g.y.to_numpy(); xx=g.x.to_numpy(); pp=g.paper_value.to_numpy()
    for c,t in enumerate(THRESHOLDS):
        safe=g[f'safe_{t:.3f}'].to_numpy(); out=im.copy(); out[yy[safe],xx[safe]]=np.clip(np.rint(pp[safe]),0,255).astype(np.uint8); axes[r,c].imshow(out,cmap='gray',vmin=0,vmax=255); axes[r,c].set_title(f'{page} {t:.3f}
{safe.sum():,}'); axes[r,c].axis('off')
plt.tight_layout(); plt.savefig(REVIEW/'top_increment_pages_threshold_ladder.png',dpi=110,bbox_inches='tight'); plt.close(fig)

## 12. Decision-oriented summary

In [ ]:
def safety_status(t):
    if truth is None:return 'UNAVAILABLE'
    r=truth_summary.loc[truth_summary.threshold==t].iloc[0]
    return f"false_remove={int(r.false_remove_print):,}; rate={r.false_remove_rate:.6f}; Wilson95={r.wilson95_upper_false_remove:.6f}"

rows=[]
for _,r in summary.iterrows():
    rows.append({'threshold':r.threshold,'selected_pixels':int(r.selected_pixels),'coverage':r.coverage,'increment_vs_0_995':int(r.incremental_pixels_vs_0_995),'pages_selected':int(r.pages_selected),'selected_components':int(r.selected_components),'top10_page_share':r.top10_page_share,'top10_component_share':r.top10_component_share,'page_rank_spearman_vs_0_995':float(stability.loc[stability.threshold==r.threshold,'spearman_vs_0_995'].iloc[0]),'source_group_status':'AVAILABLE' if source_status.get('available') else 'UNAVAILABLE','truth_safety':safety_status(r.threshold),'promotion_authority':False})
decision=pd.DataFrame(rows); decision.to_csv(TABLES/'decision_oriented_threshold_summary.csv',index=False)

truth_note='Truth-side safety was calculated from the resolved compatible truth table.' if truth is not None else 'No compatible page/y/x truth table was resolved. Coverage, concentration and stability are measured, but safety is not established by this notebook.'
lines=['NOTEBOOK 63 — FROZEN 59R2 DIAGNOSTIC OPERATING-POINT ENVELOPE','',f'Source Notebook 61: {NB61}',f'Frozen policy: {policy.get("policy_id")}',f'Pages: {cohort.page.nunique()}',f'Rows: {len(cohort):,}','',decision.to_string(index=False),'',truth_note,'','STATUS: post-outcome diagnostic sensitivity only.','No threshold in this notebook is a frozen challenger, independent validation or production authorisation.']
(RUN/'executive_summary.txt').write_text('
'.join(lines)+'
'); print('
'.join(lines))

## 13. Identity, audit and curated upload package

In [ ]:
environment={'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}
identity={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'Notebook63-59R2-diagnostic-operating-point-envelope-v1','development_sensitivity_only':True,'promotion_authority':False,'retrained':False,'refitted':False,'recalibrated':False,'source_notebook61':str(NB61),'source_parquet':str(PARQUET_PATH),'source_parquet_sha256':sha256_file(PARQUET_PATH),'frozen_59r2':str(R59R2),'policy_id':policy.get('policy_id'),'compatibility_id':policy.get('compatibility_id'),'thresholds':THRESHOLDS,'coherence':contract_record,'pages':cohort.page.nunique(),'rows':len(cohort),'source_group_resolution':source_status,'truth_resolution':truth_status,'software':environment}
atomic_json(RUN/'experiment_identity.json',identity); atomic_json(RUN/'software_environment.json',environment)

curated=[RUN/'executive_summary.txt',RUN/'experiment_identity.json',RUN/'software_environment.json',AUDIT/'resolved_frozen_contract.json',AUDIT/'frozen_replay.json',AUDIT/'resolved_page_images.csv',AUDIT/'source_group_resolution.json',AUDIT/'truth_resolution.json',TABLES/'threshold_envelope_summary.csv',TABLES/'decision_oriented_threshold_summary.csv',TABLES/'incremental_threshold_transitions.csv',TABLES/'page_rank_stability.csv',TABLES/'threshold_results_by_page.csv',TABLES/'threshold_results_by_source_group.csv',TABLES/'visual_review_queue.csv',CHARTS/'threshold_envelope_dashboard.png',CHARTS/'page_concentration_curves.png',FOCUS/'page258_threshold_ladder.png',REVIEW/'top_increment_pages_threshold_ladder.png']
if truth is not None: curated += [AUDIT/'truth_join_audit.json',TABLES/'truth_safety_summary.csv',TABLES/'truth_safety_by_page.csv',TABLES/'truth_safety_by_source_group.csv',CHARTS/'truth_safety_dashboard.png']
else: curated += [TABLES/'truth_safety_unavailable.csv']
curated += sorted(FOCUS.glob('page258_threshold_*.png'))
missing=[str(p) for p in curated if not p.is_file()]; require(not missing,f'Missing curated artefacts: {missing}')
manifest=[]
for src in curated:
    rel=src.relative_to(RUN); dst=UPLOAD/rel; dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst); require(sha256_file(src)==sha256_file(dst),f'Copy mismatch: {rel}'); manifest.append({'relative_path':str(rel),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
pd.DataFrame(manifest).sort_values('relative_path').to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'README.txt').write_text('Notebook 63 diagnostic threshold-envelope package.
The large source parquet and full working component table are not included.
Alternative thresholds are post-outcome diagnostics only.
Truth-side safety is reported only when an exact compatible truth table was resolved.
')
(UPLOAD/'upload_file_list.txt').write_text('
'.join(sorted(str(p.relative_to(UPLOAD)) for p in UPLOAD.rglob('*') if p.is_file()))+'
')
zip_path=RUN/'Notebook63_59R2_Diagnostic_Operating_Point_Envelope_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(UPLOAD.rglob('*')):
        if p.is_file(): zf.write(p,p.relative_to(UPLOAD))
with zipfile.ZipFile(zip_path) as zf: require(zf.testzip() is None,'ZIP integrity failed.')
print('Upload ZIP:',zip_path); print('SHA-256:',sha256_file(zip_path))